# CF0360 all-species DA, May 2019: ObsFcstAna maps and time series

This notebook makes maps and daily time series of observation-space statistics for the
cubed-sphere CF0360 GEOSldas all-species DA experiment over May 2019. It compares that
experiment with a 6-member open loop (OL6) that saw the same observations but did not
assimilate them.

| Name here | Experiment | What it is |
|---|---|---|
| `allsp` | `M21C_test_CF0360_allsp_superob025ml_OL6sc_xc03125` | 24-member DA: SMOS, SMAP, MODIS SCF, CYGNSS and 0.25° H SAF super-obs, all assimilated |
| `OL6` | `M21C_test_CF0360_OL6_superob025ml_clim` | 6-member open loop, all species monitored with `scale = .false.` (the run used to build the H SAF super-ob climatology) |

**Figures made:**

1. Relative change in O-F standard deviation, `(allsp − OL6) / OL6`, per sensor (the headline map, same metric as `M21C_ls/notebooks/LS_ofa_figures_refactor_20260327.ipynb`).
2. O-F mean (bias) in allsp.
3. Standard deviation of the normalized innovations in allsp (filter consistency; 1 = consistent).
4. Number of observations per map cell (context for 1–3).
5. Daily O-F mean and standard deviation, OL6 vs allsp.

**Where the maps are drawn** (section 4): each sensor is mapped on **its own observation grid**, so every coloured
cell is exactly one obs footprint and every grey gap is real (no obs, or too few). SMOS, SMAP and CYGNSS use the
EASE2 36 km grid, H SAF super-obs use 0.25° lat/lon cells, and MODIS SCF, which covers essentially every tile,
is drawn in tile space. For the tile/grid geometry itself see `cf0360_grid_and_tiles.ipynb`.

**How to run:** on Discover, with the GEOSpyD Python 3.13 environment
(`/usr/local/other/GEOSpyD/25.3.1-0/2025-10-07/envs/py3.13`), which has netCDF4, pandas,
scipy and cartopy. Reading and matching the ObsFcstAna files takes about 2 minutes and needs about 9 GB of memory. The result is
cached (see `CACHE_DIR`), so later runs only redo the plots. Set `FORCE_RECOMPUTE = True` to rebuild the cache.

> **Temporary choice: `NMIN = 10`.** The repo convention (e.g. `LS_ofa_figures_refactor_20260327.ipynb`)
> masks any map cell with fewer than **20** obs. One month of data gives many cells fewer than 20 SMAP, SMOS or
> CYGNSS obs, so this notebook uses **10**. That choice applies **only to this one-month analysis**. Do not copy it
> into other notebooks or shared code. It lives in a single clearly labelled config variable so it is easy to find and revert.

## 1. Imports

Plain scientific stack plus cartopy for the Robinson maps and `scipy.spatial.cKDTree` for the
tile → raster lookup used for MODIS (section 4).

In [ ]:
import glob
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import netCDF4 as nc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.spatial import cKDTree

%matplotlib inline

## 2. Configuration

Everything you might want to change is in this one cell: which runs, which dates, where the caches and
figures go, which grid each sensor is mapped on, and the (temporary) minimum-obs threshold.

- `ROOT` is where the experiment directories live. `/gpfsm/dnb06/projects/p284` and
  `/discover/nobackup/projects/land_da` point to the same place.
- `CACHE_DIR` sits **outside** the git repo, because the cached statistics are large binary files.
- `FIG_DIR` sits inside the project, but `*.png` is git-ignored repo-wide, so figures are not committed.

In [ ]:
def find_repo_root(start: Path) -> Path:
    """Walk up from the notebook's directory until we find the git repo root."""
    for p in [start] + list(start.parents):
        if (p / '.git').exists():
            return p
    return start

REPO_ROOT = find_repo_root(Path.cwd())

# --- Experiments --------------------------------------------------------------------------
ROOT = Path('/gpfsm/dnb06/projects/p284/M21C_testing')
DOMAIN = 'CF0360x6C_GLOBAL'                  # the output/<DOMAIN>/ subdirectory name
RUNS = {
    'allsp': 'M21C_test_CF0360_allsp_superob025ml_OL6sc_xc03125',   # DA run
    'OL6':   'M21C_test_CF0360_OL6_superob025ml_clim',              # open-loop reference
}
DA_RUN = 'allsp'   # its (scaled) obs are used as "O" for BOTH runs -- see section 5
OL_RUN = 'OL6'

# Analysis period: ObsFcstAna cycles whose date (yyyymmdd) is within [START, END], inclusive.
START, END = '20190501', '20190531'

# --- Where things go ----------------------------------------------------------------------
CACHE_DIR = ROOT / 'analysis_M21C_land' / 'allsp_vs_OL6_may2019'
FIG_DIR = REPO_ROOT / 'projects' / 'M21C_land' / 'figures' / 'allsp_vs_OL6_may2019'
FORCE_RECOMPUTE = False      # True = re-read all ObsFcstAna files even if a cache exists

# --- Minimum number of obs per map cell -------------------------------------------------------
NMIN_REPO_DEFAULT = 20       # the value used everywhere else in this repo
# !!! TEMPORARY, specific to this one-month (May 2019) analysis !!!
# One month gives too few SMAP/SMOS/CYGNSS obs per cell for Nmin = 20 (see the table in
# section 6). Revert to NMIN_REPO_DEFAULT for multi-month periods, and do not copy this value
# into other notebooks or shared code.
NMIN = 10

# --- Sensors ----------------------------------------------------------------------------------
# Group names come from projects/ascat_da/scripts/compare_thin_superob_ofa.py (GROUPS), which
# maps each obs_param 'descr' prefix to a sensor group.
GROUP_ORDER = ['SMOS Tb', 'SMAP Tb', 'HSAF SM', 'CYGNSS SM', 'MODIS SCF']
GROUP_UNITS = {'SMOS Tb': 'K', 'SMAP Tb': 'K', 'HSAF SM': 'm³/m³', 'CYGNSS SM': 'm³/m³',
               'MODIS SCF': '-'}

# Grid each sensor's statistics are computed and mapped on (section 4):
#   'ease_m36' : EASE2 global 36 km grid -- SMOS/SMAP L1C Tb and CYGNSS obs sit exactly on its cell centres
#   'deg025'   : 0.25 deg lat/lon cells  -- the H SAF super-ob cells (obs are at the mean location of the
#                                           raw obs inside each cell, so they are binned by the cell they fall in)
#   'tile'     : model tiles             -- MODIS SCF obs cover essentially every tile
GROUP_GRID = {'SMOS Tb': 'ease_m36', 'SMAP Tb': 'ease_m36', 'CYGNSS SM': 'ease_m36',
              'HSAF SM': 'deg025', 'MODIS SCF': 'tile'}

# Relative change (allsp - OL6)/OL6 is undefined where the OL O-F std is ~0 (e.g. MODIS SCF cells that
# are snow-free or fully snow-covered all month, where O = F = 0 or 1). Such cells are left out of map
# 8a, and the number left out is printed in each panel title.
MIN_OL_STD = {'SMOS Tb': 0.1, 'SMAP Tb': 0.1, 'HSAF SM': 0.001, 'CYGNSS SM': 0.001, 'MODIS SCF': 0.01}

# --- Maps -------------------------------------------------------------------------------------
MAP_LAT_MIN = -60.0          # repo map convention: nothing south of 60S (Antarctica)
RASTER_RES_DEG = 0.1         # tile-space raster resolution (MODIS only)
MAX_TILE_DIST_KM = 30.0      # tile-space raster pixels farther than this from any land tile = ocean

CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('repo root :', REPO_ROOT)
print('cache dir :', CACHE_DIR)
print('figure dir:', FIG_DIR)
print(f'NMIN = {NMIN}  (TEMPORARY; repo default is {NMIN_REPO_DEFAULT})')

## 3. Tiles and sensor groups

### 3a. Land tiles

GEOSldas works on **tiles**, which are pieces of hydrological catchments cut by the cubed-sphere grid cells
(see `cf0360_grid_and_tiles.ipynb`). The tile file (`*.domain` in `rc_out/`) lists every tile (land, lake, ice,
ocean). The land tiles (`typ == 100`) come first and are the model's tile space, in the same order as
the tile dimension of the model output files.

In ObsFcstAna files, `tilenum` is a **1-based** index into these land tiles. That was checked against the
`lat`/`lon` of the `catch_progn_incr` files: with `tilenum - 1`, obs and tile positions agree to within
one obs cell.

We keep each tile's centre of mass (for the MODIS raster) and its area (for area-weighted means).

In [ ]:
domain_file = ROOT / RUNS[DA_RUN] / 'output' / DOMAIN / 'rc_out' / 'CF0360x6C_CF0360x6C-Pfafstetter.nc4.domain'
with nc.Dataset(domain_file) as d:
    typ = d['typ'][:]
    land = typ == 100
    tile_lat = np.asarray(d['com_lat'][:][land], dtype=np.float64)
    tile_lon = np.asarray(d['com_lon'][:][land], dtype=np.float64)
    tile_area = np.asarray(d['area'][:][land], dtype=np.float64)    # radian^2; only relative size matters
N_TILE = tile_lat.size
print(f'{N_TILE:,} land tiles (of {typ.size:,} tiles in the tile file)')

### 3b. Species → sensor group

The ObsFcstAna `species` index counts, in order, the `obs_param_nml` entries that were read (those with
`assim` or `getinnov` true), so its meaning depends on each run's namelist. `species_groups()` (reused from
`projects/ascat_da`) reads the run's `LDASsa_SPECIAL_inputs_ensupd.nml` and returns
`{species index: group name}`. We require both runs to have the **same** mapping, because obs are matched by
species index in section 5.

In [ ]:
sys.path.insert(0, str(REPO_ROOT / 'projects' / 'ascat_da' / 'scripts'))
from compare_thin_superob_ofa import species_groups

groups = {name: species_groups(ROOT / exp) for name, exp in RUNS.items()}
assert groups[DA_RUN] == groups[OL_RUN], 'species order differs between runs'
SPECIES_TO_GROUP = groups[DA_RUN]
print(SPECIES_TO_GROUP)

## 4. Map grids: each sensor on its own obs grid

**Why not tile space for everything?** Each observation is assigned to **one** tile (the tile
it falls in). SMOS, SMAP and CYGNSS obs come on the 36 km EASE2 grid and H SAF super-obs on a 0.25° grid,
but a CF0360 grid cell (~28 km) already holds several tiles. So only about one tile in several ever receives
obs of a given sensor, and a per-tile map is mostly empty specks even where every obs cell is full.

So statistics are accumulated on the **obs grid** instead: every obs is put in the grid cell it belongs to,
and each coloured cell on the map is one real obs footprint. A grey cell has no obs, or fewer than `NMIN`.
Nothing is interpolated or filled.

- **EASE2 M36** (964 × 406 cells, equal area): column edges are uniform in longitude. Row edges are the
  midpoints between the row-centre latitudes read from the repo's `ease_grids/` files (outer edges at ±90°).
  SMOS/SMAP Tb and CYGNSS obs sit exactly on the cell centres (checked when this notebook was written).
- **0.25° lat/lon** (1440 × 720 cells): the H SAF super-ob cells.
- **Tile space** (MODIS only): a 0.1° raster in which each land pixel shows its nearest land tile. MODIS obs
  hit essentially every tile, so there is no sparsity problem. The lookup is cached.

`grid_cell(grid, lat, lon)` returns the flat cell index of each obs. `GRID_SHAPE`, `GRID_EDGES` and
`GRID_WEIGHT` (the relative cell area, for area-weighted means) describe each grid.

In [ ]:
# --- EASE2 M36 --------------------------------------------------------------------------------
EASE_PATH = REPO_ROOT / 'common' / 'python' / 'plotting' / 'ease_grids'
m36_lat_ctr = np.fromfile(EASE_PATH / 'EASE2_M36km.lats.964x406x1.double').reshape(406, 964)[:, 0]  # N -> S
m36_lat_ctr = m36_lat_ctr[::-1]                                              # make it S -> N
m36_lat_edges = np.concatenate([[-90.0], 0.5 * (m36_lat_ctr[:-1] + m36_lat_ctr[1:]), [90.0]])
m36_lon_edges = np.linspace(-180.0, 180.0, 965)

# --- 0.25 deg -----------------------------------------------------------------------------------
d025_lat_edges = np.linspace(-90.0, 90.0, 721)
d025_lon_edges = np.linspace(-180.0, 180.0, 1441)

GRID_EDGES = {'ease_m36': (m36_lat_edges, m36_lon_edges), 'deg025': (d025_lat_edges, d025_lon_edges)}
GRID_SHAPE = {g: (e[0].size - 1, e[1].size - 1) for g, e in GRID_EDGES.items()}
GRID_SHAPE['tile'] = (N_TILE,)


def grid_cell(grid, lat, lon, tilenum=None):
    """Flat index of the map cell each obs belongs to (for 'tile': tilenum - 1)."""
    if grid == 'tile':
        return tilenum - 1
    lat_e, lon_e = GRID_EDGES[grid]
    i = np.clip(np.searchsorted(lat_e, lat, side='right') - 1, 0, lat_e.size - 2)
    j = np.clip(np.searchsorted(lon_e, lon, side='right') - 1, 0, lon_e.size - 2)
    return i * (lon_e.size - 1) + j


# Relative cell area, for area-weighted means over cells (EASE2 is equal-area)
GRID_WEIGHT = {
    'ease_m36': np.ones(GRID_SHAPE['ease_m36']).ravel(),
    'deg025': np.repeat(np.cos(np.radians(0.5 * (d025_lat_edges[:-1] + d025_lat_edges[1:]))), 1440),
    'tile': tile_area,
}
GRID_CTR_LAT = {
    'ease_m36': np.repeat(m36_lat_ctr, 964),
    'deg025': np.repeat(0.5 * (d025_lat_edges[:-1] + d025_lat_edges[1:]), 1440),
    'tile': tile_lat,
}

### 4b. Tile-space raster (MODIS)

A 0.1° lat/lon raster north of 60S. Each pixel gets the index of the **nearest land-tile centre**, found with a
KD-tree on 3-D unit vectors (correct across the dateline and near the poles). Pixels more than
`MAX_TILE_DIST_KM` from any tile centre are ocean (−1).

In [ ]:
def lonlat_to_xyz(lon_deg, lat_deg):
    """Unit-sphere Cartesian coordinates, so that Euclidean nearest = great-circle nearest."""
    lon, lat = np.radians(lon_deg), np.radians(lat_deg)
    return np.column_stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])

tr_lon_edges = np.arange(-180.0, 180.0 + RASTER_RES_DEG / 2, RASTER_RES_DEG)
tr_lat_edges = np.arange(MAP_LAT_MIN, 90.0 + RASTER_RES_DEG / 2, RASTER_RES_DEG)
tr_lon_ctr = 0.5 * (tr_lon_edges[:-1] + tr_lon_edges[1:])
tr_lat_ctr = 0.5 * (tr_lat_edges[:-1] + tr_lat_edges[1:])

lookup_file = CACHE_DIR / f'tile_raster_lookup_{RASTER_RES_DEG:g}deg_{MAX_TILE_DIST_KM:g}km.npy'
if lookup_file.exists():
    LOOKUP = np.load(lookup_file)
else:
    t0 = time.time()
    tree = cKDTree(lonlat_to_xyz(tile_lon, tile_lat))
    LON2, LAT2 = np.meshgrid(tr_lon_ctr, tr_lat_ctr)
    chord, idx = tree.query(lonlat_to_xyz(LON2.ravel(), LAT2.ravel()))
    dist_km = 2 * 6371.0 * np.arcsin(np.clip(chord / 2, 0, 1))      # chord length -> great-circle km
    idx[dist_km > MAX_TILE_DIST_KM] = -1
    LOOKUP = idx.reshape(LAT2.shape).astype(np.int32)
    np.save(lookup_file, LOOKUP)
    print(f'built lookup in {time.time() - t0:.0f} s')
print(f'tile raster {LOOKUP.shape}, land pixels {np.mean(LOOKUP >= 0):.1%}')


def to_map(grid, values):
    """Turn a flat per-cell (or per-tile) array into (lat_edges, lon_edges, 2-D array) for pcolormesh."""
    if grid == 'tile':
        out = np.full(LOOKUP.shape, np.nan)
        ok = LOOKUP >= 0
        out[ok] = values[LOOKUP[ok]]
        return tr_lat_edges, tr_lon_edges, out
    lat_e, lon_e = GRID_EDGES[grid]
    return lat_e, lon_e, values.reshape(GRID_SHAPE[grid])

## 5. Read ObsFcstAna files and match the obs of the two runs

**Why matching is needed.** OL6 ran with `scale = .false.` for every species, so its `obs` field holds
**unscaled** observations. Those are in a different space from the model (Tb climatology, or H SAF degree
of saturation vs volumetric soil moisture). allsp stores **scaled** obs. To compare like with like, we use
the allsp obs as "O" for **both** runs, so only the forecasts differ:

- `omf_allsp = obs_allsp − fcst_allsp`
- `omf_OL6   = obs_allsp − fcst_OL6`

An observation is matched between runs on `(cycle, species, tilenum, lat, lon)`. This is the same method as
`M21C_testing/analysis_allsp_vs_OL6_may/compare_common_obs.py`, which found about 20.7 M common obs for May.

**Extra allsp-only quantities** (used for filter-consistency diagnostics, **assimilated obs only**, `assim_flag == 1`):

- normalized innovation `(O − F) / sqrt(obsvar + fcstvar)`: its std should be about 1 if the assumed obs error
  and the ensemble forecast error variance are right; above 1 means they are too small.
- `O − A`, the analysis residual.

In [ ]:
KEYS = ['cycle', 'species', 'tilenum', 'lat', 'lon']


def ofa_files(exp):
    """All ObsFcstAna files of a run within [START, END], sorted by cycle."""
    pattern = ROOT / exp / 'output' / DOMAIN / 'ana' / 'ens_avg' / 'Y*' / 'M*' / '*ldas_ObsFcstAna.*.nc4'
    files = sorted(glob.glob(str(pattern)))
    return [f for f in files if START <= os.path.basename(f).split('.')[-2][:8] <= END]


def read_run(exp, full):
    """Read a run's ObsFcstAna files into one DataFrame.

    full=False: only the matching keys and fcst (enough for the open loop).
    full=True : also obs, obsvar, fcstvar, ana and assim_flag (for the DA run).
    lat/lon are rounded to 1e-4 deg so that the same obs matches exactly across runs.
    """
    frames = []
    for f in ofa_files(exp):
        cycle = os.path.basename(f).split('.')[-2]          # e.g. 20190515_0300z
        with nc.Dataset(f) as d:
            if d.dimensions['n_obs'].size == 0:
                continue
            g = lambda v: np.ma.filled(d[v][:].astype(np.float32), np.nan)
            cols = {'cycle': cycle,
                    'species': d['species'][:].astype(np.int16),
                    'tilenum': d['tilenum'][:].astype(np.int32),
                    'lat': np.round(g('lat'), 4), 'lon': np.round(g('lon'), 4),
                    'fcst': g('fcst')}
            if full:
                cols |= {'obs': g('obs'), 'obsvar': g('obsvar'), 'fcstvar': g('fcstvar'),
                         'ana': g('ana'), 'assim': d['assim_flag'][:].astype(np.int8)}
            frames.append(pd.DataFrame(cols))
    df = pd.concat(frames, ignore_index=True).drop_duplicates(KEYS)
    print(f'  {exp}: {df.cycle.nunique()} cycles, {len(df):,} obs')
    return df

### 5a. Reduce to per-cell and per-day statistics (cached)

We never keep the 20 M matched obs around. We reduce them straight away to **sums** per map cell
(on each sensor's grid from section 4) and per (sensor, day), from which any mean or standard deviation can
be computed:

- `N` (count), `S` (sum of x) and `Q` (sum of x²), giving mean = S/N and variance = (Q − S²/N)/(N − 1).

Sums are accumulated with `np.bincount` in float64. They are saved to `CACHE_DIR/ofa_cell_moments.npz`
(keys like `SMAP Tb|omf_da_N`) and `ofa_daily.csv`.

In [ ]:
cell_cache = CACHE_DIR / 'ofa_cell_moments.npz'
daily_cache = CACHE_DIR / 'ofa_daily.csv'
QUANTITIES = ['omf_da', 'omf_ol', 'norm_da', 'oma_da']


def cell_moments(cell, x, size):
    """N, sum and sum of squares of x per map cell, ignoring NaN."""
    ok = np.isfinite(x)
    xx = x[ok].astype(np.float64)
    return (np.bincount(cell[ok], minlength=size),
            np.bincount(cell[ok], weights=xx, minlength=size),
            np.bincount(cell[ok], weights=xx * xx, minlength=size))


if cell_cache.exists() and daily_cache.exists() and not FORCE_RECOMPUTE:
    print('using cached statistics in', CACHE_DIR)
else:
    t0 = time.time()
    da = read_run(RUNS[DA_RUN], full=True)
    ol = read_run(RUNS[OL_RUN], full=False).rename(columns={'fcst': 'fcst_ol'})
    m = da.merge(ol, on=KEYS)                  # inner join = obs present in both runs
    del da, ol
    m = m.dropna(subset=['obs', 'fcst', 'fcst_ol'])
    m['group'] = m.species.map(SPECIES_TO_GROUP)
    m = m[m.group.isin(GROUP_ORDER)]
    print(f'matched obs: {len(m):,}  ({time.time() - t0:.0f} s)')

    # Per-obs quantities (see section 5 for definitions)
    m['omf_da'] = m.obs - m.fcst
    m['omf_ol'] = m.obs - m.fcst_ol
    assim = m.assim == 1
    m['norm_da'] = np.where(assim, m.omf_da / np.sqrt(m.obsvar + m.fcstvar), np.nan)
    m['oma_da'] = np.where(assim, m.obs - m.ana, np.nan)

    # Per-cell sums, each sensor on its own grid
    arrays = {}
    for g in GROUP_ORDER:
        x = m[m.group == g]
        grid = GROUP_GRID[g]
        cell = grid_cell(grid, x.lat.to_numpy(), x.lon.to_numpy(), x.tilenum.to_numpy())
        size = int(np.prod(GRID_SHAPE[grid]))
        for q in QUANTITIES:
            n, s, sq = cell_moments(cell, x[q].to_numpy(), size)
            arrays |= {f'{g}|{q}_N': n, f'{g}|{q}_S': s, f'{g}|{q}_Q': sq}
    np.savez_compressed(cell_cache, **arrays)

    # Per-day statistics (all cells together)
    m['day'] = m.cycle.str[:8]
    daily = m.groupby(['group', 'day']).agg(
        N=('omf_da', 'size'),
        omf_da_mean=('omf_da', 'mean'), omf_da_std=('omf_da', 'std'),
        omf_ol_mean=('omf_ol', 'mean'), omf_ol_std=('omf_ol', 'std'),
        N_assim=('norm_da', 'count'), norm_da_std=('norm_da', 'std'),
        oma_da_std=('oma_da', 'std')).reset_index()
    daily.to_csv(daily_cache, index=False)
    del m
    print(f'cached in {time.time() - t0:.0f} s')

MOM = dict(np.load(cell_cache))
DAILY = pd.read_csv(daily_cache, dtype={'day': str})
DAILY['date'] = pd.to_datetime(DAILY.day)

## 6. Per-cell statistics and the Nmin mask

`cell_stats` turns the sums into per-cell N, mean and std for one sensor and quantity. Cells with fewer than
`NMIN` obs get NaN (drawn grey). The table first shows **why** `NMIN` is lowered for this analysis: the share
of cells with at least 10 and at least 20 obs for each sensor over the month.

In [ ]:
def cell_stats(group, q, nmin=NMIN):
    """(N, mean, std) per map cell of one sensor group; mean/std are NaN where N < nmin."""
    n, s, sq = MOM[f'{group}|{q}_N'], MOM[f'{group}|{q}_S'], MOM[f'{group}|{q}_Q']
    with np.errstate(invalid='ignore', divide='ignore'):
        mean = s / n
        var = (sq - s * s / n) / (n - 1)
    std = np.sqrt(np.clip(var, 0, None))
    bad = n < nmin
    mean[bad] = np.nan
    std[bad] = np.nan
    return n, mean, std


STATS = {g: {q: cell_stats(g, q) for q in QUANTITIES} for g in GROUP_ORDER}

rows = []
for g in GROUP_ORDER:
    n = STATS[g]['omf_da'][0]
    has = n > 0
    rows.append({'group': g, 'map grid': GROUP_GRID[g], 'cells with obs': int(has.sum()),
                 'median N per cell': float(np.median(n[has])),
                 f'share N>={NMIN}': np.mean(n[has] >= NMIN),
                 f'share N>={NMIN_REPO_DEFAULT}': np.mean(n[has] >= NMIN_REPO_DEFAULT)})
pd.DataFrame(rows).style.format({f'share N>={NMIN}': '{:.0%}', f'share N>={NMIN_REPO_DEFAULT}': '{:.0%}',
                                 'cells with obs': '{:,}', 'median N per cell': '{:.0f}'})

## 7. Map helper

`plot_group_maps` draws one Robinson panel per sensor group (3×2 layout, the 6th panel left empty), following
the repo map conventions: south of 60S cut off, grey land underneath (so land cells with no value show grey),
thin coastlines, and a **discrete** colour scale. For signed quantities the scale is `RdBu_r` with a white bin
centred on zero (or on 1 for normalized std).

- `values` is a dict `{group: flat array on that group's grid}`.
- `levels` is either one list for all panels (shared colour bar at the bottom) or a dict per group (one colour
  bar under each panel, when units differ between sensors).
- The panel title gives the number of cells (or tiles) with a value and a summary over them: the
  **area-weighted mean** by default, or the **median** (`summary='median'`) for ratios, where a few cells with a
  small denominator would dominate a mean. `title_extra` adds text per panel (e.g. how many cells were left out).
- `pcolormesh(..., rasterized=True)` keeps saved figures small.

In [ ]:
def plot_group_maps(values, title, cbar_label, levels, cmap='RdBu_r', fname=None, extend='both',
                    summary='mean', title_extra=None):
    shared = not isinstance(levels, dict)
    fig, axs = plt.subplots(3, 2, figsize=(15, 13.5), subplot_kw={'projection': ccrs.Robinson()},
                            constrained_layout=True)
    axs = axs.ravel()
    mesh = None
    for i, g in enumerate(GROUP_ORDER):
        ax = axs[i]
        grid = GROUP_GRID[g]
        lev = levels if shared else levels[g]
        cm = plt.get_cmap(cmap)
        norm = mcolors.BoundaryNorm(lev, cm.N, extend=extend)
        ax.set_global()
        ax.set_extent([-180, 180, MAP_LAT_MIN, 90], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
        lat_e, lon_e, field = to_map(grid, values[g])
        mesh = ax.pcolormesh(lon_e, lat_e, field, cmap=cm, norm=norm,
                             transform=ccrs.PlateCarree(), rasterized=True, zorder=1)
        ax.coastlines(linewidth=0.3, zorder=2)
        v, w = values[g], GRID_WEIGHT[grid]
        ok = np.isfinite(v) & (GRID_CTR_LAT[grid] >= MAP_LAT_MIN)
        if not ok.any():
            stat = np.nan
        elif summary == 'median':
            stat = np.median(v[ok])
        else:
            stat = np.sum(v[ok] * w[ok]) / np.sum(w[ok])
        unit_word = 'tiles' if grid == 'tile' else 'cells'
        label = 'median' if summary == 'median' else 'area-wtd mean'
        extra = f', {title_extra[g]}' if title_extra else ''
        ax.set_title(f'{g} [{grid}]  ({ok.sum():,} {unit_word}, {label} {stat:.3g}{extra})', fontsize=11)
        if not shared:
            cb = fig.colorbar(mesh, ax=ax, orientation='horizontal', shrink=0.7, pad=0.02)
            cb.set_label(f'{cbar_label} [{GROUP_UNITS[g]}]')
    axs[-1].set_visible(False)
    if shared:
        cb = fig.colorbar(mesh, ax=axs[:-1].tolist(), orientation='horizontal', shrink=0.5, pad=0.02)
        cb.set_label(cbar_label)
    fig.suptitle(title, fontsize=15)
    if fname:
        fig.savefig(FIG_DIR / fname, dpi=150)
    plt.show()


PERIOD = f'{START}–{END}'
NOTE_N = f'cells with N ≥ {NMIN} obs'

## 8. Maps

### 8a. Relative change in O-F standard deviation, `(allsp − OL6) / OL6`

The headline metric, same as the repo's M36 figures. Both runs are scored against the same (scaled) obs, so a
**negative (blue) value means the DA forecast fits the obs better** than the open loop. This is not
independent verification, because allsp assimilates all of these obs. But O-F uses the *forecast*, made before
that cycle's obs were assimilated.

Cells where the OL6 O-F std is below `MIN_OL_STD` are **left out**, because the ratio is undefined there. Their
number is given in each panel title. The panel titles give the **median** over cells.

In [ ]:
rel_change, n_dropped = {}, {}
for g in GROUP_ORDER:
    std_da, std_ol = STATS[g]['omf_da'][2], STATS[g]['omf_ol'][2]
    small = std_ol < MIN_OL_STD[g]              # NaN compares False, so only real small values count
    with np.errstate(invalid='ignore', divide='ignore'):
        r = 100 * (std_da - std_ol) / std_ol
    r[small] = np.nan
    rel_change[g] = r
    n_dropped[g] = f'{int(small.sum()):,} left out (OL std < {MIN_OL_STD[g]:g})'

plot_group_maps(rel_change,
                f'(allsp − OL6) / OL6 O-F std, {PERIOD}  ({NOTE_N})',
                'relative change in O-F std [%]  (negative = DA better)',
                levels=[-40, -30, -20, -10, -5, -2, 2, 5, 10, 20, 30, 40],
                fname='map_omf_std_relchange.png', summary='median', title_extra=n_dropped)

### 8b. O-F mean (bias) in allsp

Obs minus DA forecast, averaged per cell. **Positive = obs higher than the forecast** (for Tb: the model is too
cold; for soil moisture: the model is too dry). Each sensor has its own units and scale.

In [ ]:
bias_levels = {
    'SMOS Tb':   [-10, -6, -4, -2, -1, 1, 2, 4, 6, 10],
    'SMAP Tb':   [-10, -6, -4, -2, -1, 1, 2, 4, 6, 10],
    'HSAF SM':   [-0.05, -0.03, -0.02, -0.01, -0.005, 0.005, 0.01, 0.02, 0.03, 0.05],
    'CYGNSS SM': [-0.05, -0.03, -0.02, -0.01, -0.005, 0.005, 0.01, 0.02, 0.03, 0.05],
    'MODIS SCF': [-0.3, -0.2, -0.1, -0.05, -0.02, 0.02, 0.05, 0.1, 0.2, 0.3],
}
plot_group_maps({g: STATS[g]['omf_da'][1] for g in GROUP_ORDER},
                f'allsp O-F mean, {PERIOD}  ({NOTE_N})', 'O-F mean',
                levels=bias_levels, fname='map_omf_mean_allsp.png')

### 8c. Std of the normalized innovations in allsp (assimilated obs only)

`std[(O − F) / sqrt(obsvar + fcstvar)]` per cell. For a well-tuned filter this is about **1** (white).
**Red (> 1)**: the actual O-F spread is larger than the filter assumes (obs error and/or forecast error
variance too small, so the obs are overweighted relative to reality or the ensemble is underdispersed).
**Blue (< 1)**: the assumed errors are too large. Cells with fewer than `NMIN` *assimilated* obs are grey.

In [ ]:
plot_group_maps({g: STATS[g]['norm_da'][2] for g in GROUP_ORDER},
                f'allsp std of normalized innovations, {PERIOD}  (cells with ≥ {NMIN} assimilated obs)',
                'std[(O−F)/sqrt(obsvar+fcstvar)]  (1 = consistent)',
                levels=[0.3, 0.5, 0.7, 0.8, 0.9, 1.1, 1.25, 1.5, 2, 3],
                fname='map_norm_innov_std_allsp.png')

### 8d. Number of obs per cell

Matched obs per cell over the month, **before** the `NMIN` mask (cells with 1–9 obs are drawn here, but are
grey in 8a–8c). Useful for judging where the maps above rest on few obs. Log-spaced bins, sequential colours.

In [ ]:
counts = {}
for g in GROUP_ORDER:
    n = STATS[g]['omf_da'][0].astype(float)
    counts[g] = np.where(n > 0, n, np.nan)
plot_group_maps(counts, f'Matched obs per cell, {PERIOD}  (all cells with ≥ 1 obs)', 'obs per cell',
                levels=[1, 5, 10, 20, 50, 100, 200, 500], cmap='viridis', extend='max',
                fname='map_nobs_per_cell.png')

## 9. Daily time series, OL6 (dashed) vs allsp (solid)

Global statistics per day over **all** matched obs of each sensor (no Nmin mask: each day has thousands of
obs). The layout follows the repo time-series figures: 3×2 panels, one per sensor, with each series' mean in the legend.

In [ ]:
def plot_daily(col_ol, col_da, ylabel, title, fname, zero_line=False):
    fig, axs = plt.subplots(3, 2, figsize=(15, 10), sharex=True, constrained_layout=True)
    axs = axs.ravel()
    for i, g in enumerate(GROUP_ORDER):
        ax = axs[i]
        x = DAILY[DAILY.group == g].sort_values('date')
        ax.plot(x.date, x[col_ol], '--', label=f'OL6 (mean {x[col_ol].mean():.3g})')
        ax.plot(x.date, x[col_da], '-', label=f'allsp (mean {x[col_da].mean():.3g})')
        if zero_line:
            ax.axhline(0, color='k', linestyle=':', linewidth=1)
        ax.set_title(g)
        ax.set_ylabel(f'{ylabel} [{GROUP_UNITS[g]}]')
        ax.legend(fontsize=9)
        ax.tick_params(axis='x', rotation=45)
    axs[-1].set_visible(False)
    fig.suptitle(title, fontsize=14)
    fig.savefig(FIG_DIR / fname, dpi=150)
    plt.show()


plot_daily('omf_ol_mean', 'omf_da_mean', 'O − F mean', f'Daily O-F mean, {PERIOD}', 'ts_omf_mean.png',
           zero_line=True)
plot_daily('omf_ol_std', 'omf_da_std', 'O − F std', f'Daily O-F std, {PERIOD}', 'ts_omf_std.png')

## 10. Notes and next steps

- **`NMIN = 10` is temporary** (see section 2). Revert to 20 for any longer period.
- Scaled obs are from allsp for both runs (section 5), so OL6's own O-F in *its* unscaled obs space is never used.
- Not done yet: maps of increments (mean signed profile-water increment dW, count of |dW| > 100 mm) from
  `catch_progn_incr`, and the monthly root-zone difference allsp − OL6 (OL6 has only 0.5° gridded `tavg24`
  output, so that map would be on the 0.5° grid, not in tile space).